# Lekcija 10 - AI Agent u produkciji

U ovoj lekciji naučit ćete **proizvodne obrasce** za AI agente koristeći Microsoft Agent Framework (Python). Obradit ćemo:

- **Promatranje** — dodavanje mjerenja vremena i evidentiranja interakcija agenta
- **Evaluacija** — korištenje evaluacijskog agenta za ocjenu kvalitete odgovora
- **Upravljanje troškovima** — strategije za optimizaciju tokena i odabir modela

Scenarij je **putnički agent** koji pomaže korisnicima u planiranju putovanja, s nadzorom i evaluacijom iznad toga.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Proizvodni aspekti

Prelazak AI agenata iz prototipova u produkciju zahtijeva pažnju na tri stupa:

1. **Promatranje** — Potrebna vam je vidljivost u što agent radi, koliko mu vremena treba i koje alate poziva. Bez praćenja i zapisivanja, otklanjanje pogrešaka u produkciji gotovo je nemoguće.

2. **Evaluacija** — Automatske provjere kvalitete osiguravaju da agentovi odgovori ostanu točni, potpuni i korisni tijekom vremena. Evaluacijski agent može ocijeniti odgovore prema definiranim kriterijima.

3. **Upravljanje troškovima** — Korištenje tokena izravno utječe na troškove. Strategije poput optimizacije upita, odabira modela i keširanja pomažu u održavanju troškova pod kontrolom bez kompromisa u kvaliteti.


## Izgradnja promatranog agenta

Definiramo alate za putovanje i omotamo poziv agenta s mjerenjem vremena kako bismo mogli pratiti latenciju. U produkciji biste integrirali s OpenTelemetry ili sličnim sustavom za praćenje.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Obrasci evaluacije

Uobičajeni proizvodni obrazac je korištenje drugog agenta kao **evaluatora**. Evaluator ocjenjuje odgovor primarnog agenta prema prethodno definiranim kriterijima kao što su potpunost, točnost i korisnost.

Ovo omogućava:
- Automatizirane kontrole kvalitete prije nego što odgovori dođu do korisnika
- Otkrivanje regresija kada se promijene upiti ili modeli
- Kontinuirano praćenje performansi agenta tijekom vremena


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategije upravljanja troškovima

Kontrola troškova je ključna za AI agente za proizvodnju. Evo ključnih strategija:

| Strategija | Opis |
|---|---|
| **Optimizacija prompta** | Održavajte sistemske upute sažetima. Uklonite suvišni kontekst kako biste smanjili ulazne tokene. |
| **Odabir modela** | Koristite manje, jeftinije modele (npr. GPT-4o-mini) za jednostavne zadatke poput klasifikacije ili izdvajanja, a veće modele zadržite za složeno rezoniranje. |
| **Keširanje** | Keširajte rezultate alata i česte upite kako biste izbjegli suvišne API pozive. |
| **Budžeti tokena** | Postavite limite `max_tokens` kako biste spriječili neočekivano duge odgovore. |
| **Grupiranje** | Grupišite više korisničkih upita u jedan API poziv gdje je to moguće. |

U praksi, dobro funkcionira slojeviti pristup: usmjerite jednostavne zahtjeve prema brzom, jeftinom modelu i eskalirajte samo složene upite prema sposobnijem modelu.


## Sažetak

U ovoj lekciji naučili ste kako:

1. **Dodati uočljivost** interakcijama agenata pomoću mjerenja vremena i zapisivanja, stvarajući temelje za praćenje i nadzor.
2. **Automatski ocijeniti odgovore agenata** koristeći evaluacijskog agenta koji boduje potpunost, točnost i korisnost.
3. **Upravljati troškovima** kroz optimizaciju upita, odabir modela, keširanje i budžete tokena.

Ovi obrasci za produkciju pomažu osigurati da su vaši AI agenti pouzdani, mjerljivi i ekonomični u velikom opsegu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
